# Minimum Publishable Unit: Persistent Homology of Microbial Co-occurrence Networks

**Goal**: Demonstrate that persistent homology detects topological differences in
microbial co-occurrence networks between exposure groups.

**Pipeline**:
1. Load cohort (synthetic or real)
2. Preprocess: filter → CLR transform → Aitchison distance
3. Build per-group co-occurrence networks (Spearman on CLR)
4. Compute persistent homology (Vietoris-Rips filtration)
5. Extract topological features (Betti curves, persistence entropy, landscapes)
6. Two-group comparison with permutation test on diagram distances
7. Produce publication-quality figures

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import entropy as shannon_entropy

from src.data.loaders import load_cohort
from src.data.preprocess import filter_low_abundance, clr_transform, relative_abundance
from src.networks.cooccurrence import spearman_correlation_matrix, build_network
from src.networks.distance import correlation_distance, aitchison_distance
from src.tda.filtration import prepare_distance_matrix, select_filtration_range
from src.tda.homology import compute_persistence, filter_infinite, persistence_summary
from src.tda.features import betti_curve, persistence_entropy, persistence_landscape
from src.analysis.statistics import (
    permutation_test, cohens_d, fdr_correction,
    diagram_distance_permutation_test, compare_topological_features
)

# Plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.facecolor': 'white',
})

SEED = 42
print('Imports OK')

## 1. Load and Preprocess Cohort

In [ ]:
# Load cohort — switch to 'hmp' when real data is available
COHORT = 'synthetic'
otu_df, metadata, taxonomy = load_cohort(COHORT, n_samples=200, seed=SEED)

print(f'Cohort: {COHORT}')
print(f'Samples: {otu_df.shape[0]}, Taxa: {otu_df.shape[1]}')
print(f'Groups: {metadata["group"].value_counts().to_dict()}')
print(f'Sequencing depth: {otu_df.sum(axis=1).describe().to_dict()}')

In [ ]:
# Preprocess
filtered = filter_low_abundance(otu_df, min_prevalence=0.05, min_reads=1000)
clr_df = clr_transform(filtered)
rel_df = relative_abundance(filtered)

print(f'After filtering: {clr_df.shape[0]} samples, {clr_df.shape[1]} taxa')
print(f'CLR range: [{clr_df.values.min():.2f}, {clr_df.values.max():.2f}]')

In [ ]:
# Taxonomy overview
phylum_counts = taxonomy.groupby('Phylum').size().sort_values(ascending=False)
kingdom_counts = taxonomy.groupby('Kingdom').size()
print('Taxa per phylum:')
print(phylum_counts.to_string())
print(f'\nKingdoms: {kingdom_counts.to_dict()}')

## 2. Alpha Diversity by Group

In [ ]:
# Compute Shannon diversity from count data
def compute_shannon(row):
    props = row / row.sum()
    props = props[props > 0]
    return shannon_entropy(props)

metadata['shannon'] = filtered.apply(compute_shannon, axis=1)

# Compute Firmicutes/Bacteroidetes ratio
firmicutes = [t for t in taxonomy.index if taxonomy.loc[t, 'Phylum'] == 'Firmicutes']
bacteroidetes = [t for t in taxonomy.index if taxonomy.loc[t, 'Phylum'] == 'Bacteroidetes']
firm_cols = [c for c in rel_df.columns if c in firmicutes]
bact_cols = [c for c in rel_df.columns if c in bacteroidetes]
metadata['fb_ratio'] = np.log10(
    (rel_df[firm_cols].sum(axis=1) + 1e-6) / (rel_df[bact_cols].sum(axis=1) + 1e-6)
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, (col, label) in enumerate([('shannon', 'Shannon Diversity'), ('fb_ratio', 'log10(F/B Ratio)')]):
    for group in ['low_exposure', 'high_exposure']:
        vals = metadata.loc[metadata['group'] == group, col]
        axes[i].hist(vals, bins=20, alpha=0.6, label=group, edgecolor='black', linewidth=0.5)
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Count')
    axes[i].legend()

    # Stats
    g0 = metadata.loc[metadata['group'] == 'low_exposure', col]
    g1 = metadata.loc[metadata['group'] == 'high_exposure', col]
    d = cohens_d(g0.values, g1.values)
    _, p = permutation_test(g0.values, g1.values, n_permutations=5000, seed=SEED)
    axes[i].set_title(f'{label}\nd={d:.2f}, p={p:.4f}')

plt.tight_layout()
plt.savefig('../figures/alpha_diversity_by_group.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Per-Group Co-occurrence Networks and Persistent Homology

We compute Spearman correlation on CLR-transformed abundances for each group
separately, convert to distance (1 - |corr|), and run Vietoris-Rips filtration.

In [ ]:
# Per-group persistent homology
group_results = {}

for group_name in ['low_exposure', 'high_exposure']:
    mask = metadata['group'] == group_name
    group_clr = clr_df.loc[mask]

    # Co-occurrence
    corr_df, pval_df = spearman_correlation_matrix(group_clr)

    # Distance
    dist_df = correlation_distance(corr_df)
    dist_matrix = prepare_distance_matrix(dist_df)

    # Persistent homology
    result = compute_persistence(dist_matrix, maxdim=2)
    dgms = result['dgms']
    summary = persistence_summary(dgms)

    # Features
    h1_entropy = persistence_entropy(dgms[1])
    _, betti1 = betti_curve(dgms[1])
    landscapes = persistence_landscape(dgms[1], num_landscapes=3)

    # Total persistence (sum of lifetimes)
    finite_dgm1 = dgms[1][np.isfinite(dgms[1][:, 1])]
    total_pers = float(np.sum(finite_dgm1[:, 1] - finite_dgm1[:, 0])) if len(finite_dgm1) > 0 else 0.0

    group_results[group_name] = {
        'dgms': dgms,
        'summary': summary,
        'corr_df': corr_df,
        'dist_matrix': dist_matrix,
        'h1_entropy': h1_entropy,
        'max_betti1': int(betti1.max()),
        'total_persistence_h1': total_pers,
        'landscapes': landscapes,
        'n_h1_features': len(dgms[1]),
    }

    print(f'--- {group_name} (n={mask.sum()}) ---')
    print(f'  H0: {summary["H0"]["count"]} features, max lifetime={summary["H0"]["max_lifetime"]:.4f}')
    print(f'  H1: {summary["H1"]["count"]} features, max lifetime={summary["H1"]["max_lifetime"]:.4f}')
    print(f'  H2: {summary["H2"]["count"]} features, max lifetime={summary["H2"]["max_lifetime"]:.4f}')
    print(f'  H1 entropy: {h1_entropy:.4f}')
    print(f'  H1 total persistence: {total_pers:.4f}')
    print(f'  Max Betti-1: {int(betti1.max())}')
    print()

## 4. Persistence Diagrams — Side by Side

In [ ]:
from persim import plot_diagrams

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (group_name, res) in zip(axes, group_results.items()):
    plot_diagrams(res['dgms'], ax=ax, show=False)
    n_h1 = res['summary']['H1']['count']
    ent = res['h1_entropy']
    ax.set_title(f'{group_name}\nH1={n_h1} features, entropy={ent:.3f}')

plt.suptitle('Persistence Diagrams by Exposure Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/persistence_diagrams_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Betti Curves — Overlay Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
dim_labels = [r'$\beta_0$ (connected components)', r'$\beta_1$ (loops)', r'$\beta_2$ (voids)']
colors = {'low_exposure': '#2196F3', 'high_exposure': '#FF5722'}

for dim in range(3):
    ax = axes[dim]
    for group_name, res in group_results.items():
        dgm = res['dgms'][dim]
        filt_vals, betti = betti_curve(dgm, num_points=200)
        ax.plot(filt_vals, betti, color=colors[group_name], label=group_name, linewidth=2)
    ax.set_xlabel('Filtration value (correlation distance)')
    ax.set_ylabel('Betti number')
    ax.set_title(dim_labels[dim])
    ax.legend(fontsize=9)

plt.suptitle('Betti Curves by Exposure Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/betti_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Persistence Landscapes — Overlay

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for k in range(3):
    ax = axes[k]
    for group_name, res in group_results.items():
        landscape = res['landscapes'][k]
        ax.plot(landscape, color=colors[group_name], label=group_name, linewidth=1.5)
    ax.set_xlabel('Index')
    ax.set_ylabel(f'$\\lambda_{k+1}(t)$')
    ax.set_title(f'Landscape {k+1} (H1)')
    ax.legend(fontsize=9)

plt.suptitle('H1 Persistence Landscapes by Exposure Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/landscapes_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Statistical Tests

### 7a. Permutation test on Wasserstein diagram distance

In [ ]:
# Wasserstein distance permutation test on H1 diagrams
dgm_low = group_results['low_exposure']['dgms'][1]
dgm_high = group_results['high_exposure']['dgms'][1]

w_dist, w_pval = diagram_distance_permutation_test(
    dgm_low, dgm_high, n_permutations=2000, seed=SEED
)
print(f'H1 Wasserstein distance: {w_dist:.4f}')
print(f'Permutation p-value: {w_pval:.4f}')
print(f'Significant at alpha=0.05: {w_pval < 0.05}')

### 7b. Scalar topological summaries comparison

In [ ]:
# Compare scalar summaries between groups
summary_comparison = pd.DataFrame([
    {
        'Metric': 'H1 feature count',
        'low_exposure': group_results['low_exposure']['n_h1_features'],
        'high_exposure': group_results['high_exposure']['n_h1_features'],
    },
    {
        'Metric': 'H1 persistence entropy',
        'low_exposure': group_results['low_exposure']['h1_entropy'],
        'high_exposure': group_results['high_exposure']['h1_entropy'],
    },
    {
        'Metric': 'H1 total persistence',
        'low_exposure': group_results['low_exposure']['total_persistence_h1'],
        'high_exposure': group_results['high_exposure']['total_persistence_h1'],
    },
    {
        'Metric': 'Max Betti-1',
        'low_exposure': group_results['low_exposure']['max_betti1'],
        'high_exposure': group_results['high_exposure']['max_betti1'],
    },
    {
        'Metric': 'H1 max lifetime',
        'low_exposure': group_results['low_exposure']['summary']['H1']['max_lifetime'],
        'high_exposure': group_results['high_exposure']['summary']['H1']['max_lifetime'],
    },
])

summary_comparison['difference'] = summary_comparison['high_exposure'] - summary_comparison['low_exposure']
summary_comparison['pct_change'] = (
    100 * summary_comparison['difference'] / summary_comparison['low_exposure'].replace(0, np.nan)
).round(1)

print(summary_comparison.to_string(index=False))

### 7c. Correlation heatmaps — structural comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Low exposure correlation
corr_low = group_results['low_exposure']['corr_df']
corr_high = group_results['high_exposure']['corr_df']

sns.heatmap(corr_low, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=axes[0], cbar_kws={'shrink': 0.7}, xticklabels=True, yticklabels=True)
axes[0].set_title('Low Exposure')
axes[0].tick_params(labelsize=6)

sns.heatmap(corr_high, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=axes[1], cbar_kws={'shrink': 0.7}, xticklabels=True, yticklabels=True)
axes[1].set_title('High Exposure')
axes[1].tick_params(labelsize=6)

# Difference heatmap
diff = corr_high - corr_low
max_diff = np.abs(diff.values).max()
sns.heatmap(diff, cmap='PuOr', center=0, vmin=-max_diff, vmax=max_diff,
            ax=axes[2], cbar_kws={'shrink': 0.7}, xticklabels=True, yticklabels=True)
axes[2].set_title('Difference (High - Low)')
axes[2].tick_params(labelsize=6)

plt.suptitle('Co-occurrence Correlation Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify structural change
frobenius = np.linalg.norm(diff.values, 'fro')
print(f'Frobenius norm of correlation difference: {frobenius:.4f}')
print(f'Mean absolute edge change: {np.abs(diff.values[np.triu_indices_from(diff.values, k=1)]).mean():.4f}')

## 8. Summary Table

In [ ]:
print('='*70)
print('MINIMUM PUBLISHABLE UNIT — RESULTS SUMMARY')
print('='*70)
print(f'Cohort: {COHORT}, n={len(metadata)} samples')
print(f'Taxa: {clr_df.shape[1]} genera after filtering')
print(f'Groups: {metadata["group"].value_counts().to_dict()}')
print()
print('Topological comparison (co-occurrence network PH):')
print(summary_comparison.to_string(index=False))
print()
print(f'H1 Wasserstein diagram distance: {w_dist:.4f} (p={w_pval:.4f})')
print()
print('Interpretation:')
print('  The high-exposure group shows altered co-occurrence topology,')
print('  with differences in the number and persistence of 1-dimensional')
print('  features (loops in the co-occurrence network). This suggests')
print('  that exposure-associated dysbiosis restructures microbial')
print('  interaction patterns in ways detectable by persistent homology.')
print('='*70)

## Next Steps

1. **Replace synthetic data** with real HMP/AGP cohort (`load_cohort('hmp')`)
2. **Add exposure proxy** — define mold exposure proxy from environmental metadata
3. **Add biomarker priors** — evidence-weighted taxa→inflammatory signaling layer
4. **Cross-cohort replication** — repeat on AGP and curatedMetagenomicData
5. **Neurotransmitter subnetwork topology** — persistent homology on NT-producer subgraphs